In [1]:
import os
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

with open("./key_gemini.txt", "r") as f:
    gemini_key = f.read().strip()

Settings.llm = GoogleGenAI(
    model="gemini-3.1-flash-lite-preview", 
    api_key=gemini_key
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    embed_batch_size=42
)

/Users/nguyenhoangminh/Documents/DUT_AI/.venv_ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6796.46it/s]


In [2]:
index_dir = "./storage"

In [3]:
from llama_index.core import load_index_from_storage, StorageContext

storage_context = StorageContext.from_defaults(persist_dir=index_dir)
index = load_index_from_storage(storage_context)

In [4]:
chat = index.as_chat_engine(verbose = True)

### chat interface

In [5]:
chat.reset()

In [6]:
chat.chat_history

[]

In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import HTMLResponse
import uvicorn
import nest_asyncio

# Khắc phục lỗi event loop của Jupyter 
nest_asyncio.apply()

app = FastAPI(title="Chatbot API")

# Model cho request body
class ChatRequest(BaseModel):
    message: str

# Model cho response
class ChatResponse(BaseModel):
    response: str

html_content = """
<!DOCTYPE html>
<html lang="vi">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Chatbot UI</title>
        <style>
            body { 
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                max-width: 800px; margin: 0 auto; padding: 20px; 
                background-color: #f0f2f5; 
            }
            .chat-container {
                background: #ffffff;
                border-radius: 12px;
                box-shadow: 0 4px 20px rgba(0, 0, 0, 0.08);
                padding: 25px;
            }
            h2 {
                text-align: center;
                color: #2c3e50;
                margin-top: 0;
                font-weight: 600;
            }
            #chatbox { 
                height: 450px; 
                padding: 20px; 
                overflow-y: auto; 
                margin-bottom: 20px; 
                border-radius: 8px; 
                background: #f8f9fa;
                border: 1px solid #e9ecef;
                display: flex;
                flex-direction: column;
                gap: 12px;
            }
            .msg {
                padding: 12px 18px;
                border-radius: 18px;
                max-width: 75%;
                line-height: 1.5;
                word-wrap: break-word;
            }
            .user-msg { 
                background-color: #007bff; 
                color: white; 
                align-self: flex-end;
                border-bottom-right-radius: 4px;
            }
            .bot-msg { 
                background-color: #e9ecef; 
                color: #212529; 
                align-self: flex-start;
                border-bottom-left-radius: 4px;
            }
            #input-container { display: flex; gap: 12px; }
            #message-input { 
                flex-grow: 1; 
                padding: 12px 20px; 
                border: 1px solid #ced4da; 
                border-radius: 25px; 
                font-family: inherit; 
                font-size: 15px;
                outline: none;
                transition: border-color 0.2s, box-shadow 0.2s;
            }
            #message-input:focus {
                border-color: #80bdff;
                box-shadow: 0 0 0 0.2rem rgba(0,123,255,.25);
            }
            button { 
                padding: 12px 28px; 
                cursor: pointer; 
                background: #007bff; 
                color: white; 
                border: none; 
                border-radius: 25px; 
                font-weight: 600;
                transition: background-color 0.2s, transform 0.1s;
                font-size: 15px;
            }
            button:hover { background: #0056b3; }
            button:active { transform: scale(0.98); }
        </style>
    </head>
    <body>
        <div class="chat-container">
            <h2>✨ AI Chatbot</h2>
            <div id="chatbox"></div>
            <div id="input-container">
                <input type="text" id="message-input" placeholder="Nhập tin nhắn của bạn..." onkeypress="handleKeyPress(event)">
                <button onclick="sendMessage()">Gửi</button>
            </div>
        </div>

        <script>
            async function sendMessage() {
                var inputField = document.getElementById("message-input");
                var message = inputField.value.trim();
                if (!message) return;

                var chatbox = document.getElementById("chatbox");
                chatbox.innerHTML += '<div class="msg user-msg">' + message + '</div>';
                inputField.value = "";
                chatbox.scrollTop = chatbox.scrollHeight;

                try {
                    const response = await fetch("/chat", {
                        method: "POST",
                        headers: {
                            "Content-Type": "application/json"
                        },
                        body: JSON.stringify({ message: message })
                    });
                    const data = await response.json();
                    chatbox.innerHTML += '<div class="msg bot-msg">' + data.response + '</div>';
                } catch (error) {
                    chatbox.innerHTML += '<div class="msg bot-msg" style="color:#dc3545;">Lỗi: Không thể kết nối tới server.</div>';
                }
                chatbox.scrollTop = chatbox.scrollHeight;
            }

            function handleKeyPress(event) {
                if (event.key === "Enter") {
                    sendMessage();
                }
            }
        </script>
    </body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
async def root_endpoint():
    """Trả về giao diện HTML cho Chatbot"""
    return html_content

@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    user_message = request.message
    
    try:
        # Gọi AI sinh ra câu trả lời (thay query_engine bằng tên biến thật của bạn)
        response = chat.chat(user_message)
        bot_reply = str(response)
    except Exception as e:
        bot_reply = f"Xin lỗi, bot đang gặp lỗi: {str(e)}"
    
    return ChatResponse(response=bot_reply)

if __name__ == "__main__":
    # Đổi sang cổng 8001 để tránh bị trùng lặp ở các quá trình trước đang chạy ngầm chưa tắt
    print("Mở trình duyệt tại: http://localhost:8001")
    uvicorn.run(app, host="0.0.0.0", port=8001)


INFO:     Started server process [44111]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Mở trình duyệt tại: http://localhost:8001
INFO:     127.0.0.1:56078 - "GET / HTTP/1.1" 200 OK
Condensed question: hi
INFO:     127.0.0.1:56078 - "POST /chat HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [44111]


demo

In [ ]:
### chat interface
def wrapper_chat_history(chat_history):
    wrapper_history = []
    for m in chat_history:
        # LlamaIndex ChatMessage uses .role (MessageRole enum) and .content
        role_str = m.role.value if hasattr(m.role, "value") else str(m.role)
        if role_str in ["user", "assistant"] and m.content is not None:
            wrapper_history.append({"role": role_str, "content": m.content})
    return wrapper_history

In [ ]:
wrapper_chat_history(chat.chat_history)
def converse(message, history):
    # LlamaIndex chat engine keeps its own state, so we just pass the new message
    response = chat.chat(message)
    # Gradio's ChatInterface expects returning a string response
    return str(response)

In [ ]:
import gradio as gr
try:
    demo.close()
except:
    pass
demo = gr.ChatInterface(fn=converse)
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Condensed question: Standalone question: Phân tích vai trò của tư tưởng văn hóa truyền thống Việt Nam đối với sự hình thành tư tưởng Hồ Chí Minh là gì?
